In [1]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker


In [2]:
from pathlib import Path
import json

# Hem proje kökünden hem ragas_project klasöründen çalışır.
PROJECT_DIR = Path.cwd() if (Path.cwd() / "pdfs").is_dir() else Path.cwd() / "ragas_project"
pdf_paths = sorted(str(path) for path in (PROJECT_DIR / "pdfs").glob("*.pdf"))
if not pdf_paths:
    raise FileNotFoundError(f"PDF bulunamadı: {PROJECT_DIR / 'pdfs'}")

In [3]:
docs = []

for path in pdf_paths:

    loader = PyPDFLoader(path)

    docs.extend(loader.load())

print("PDF sayısı:", len(pdf_paths))

print("Toplam sayfa:", len(docs))

for p in pdf_paths:

    print(p)

PDF sayısı: 6
Toplam sayfa: 36
/Users/azatdundar/Desktop/DeepLearning/ragas_project/pdfs/battery_storage_and_sodium_ion_batteries.pdf
/Users/azatdundar/Desktop/DeepLearning/ragas_project/pdfs/global_energy_transition_2026.pdf
/Users/azatdundar/Desktop/DeepLearning/ragas_project/pdfs/istanbul_micro_grid_case_study.pdf
/Users/azatdundar/Desktop/DeepLearning/ragas_project/pdfs/smart_grids_and_ai.pdf
/Users/azatdundar/Desktop/DeepLearning/ragas_project/pdfs/solar_energy_innovations.pdf
/Users/azatdundar/Desktop/DeepLearning/ragas_project/pdfs/wind_energy_technologies.pdf


In [4]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="bge-m3"
)

In [5]:
splitters = {
    "recursive": RecursiveCharacterTextSplitter(

        chunk_size=1000,

        chunk_overlap=100

    ),

    "semantic": SemanticChunker(

        embeddings=embeddings

    )

}

### Comparing Text Splitting Methods

Apply each text splitter to the documents and store the resulting chunks for comparison. The number of chunks generated by each method is also reported.

In [7]:
splits_by_method = {}
for name, splitter in splitters.items():
    splits = splitter.split_documents(docs)
    for index, doc in enumerate(splits):
        doc.metadata["chunk_id"] = f"{name}-{index:04d}"
    splits_by_method[name] = splits
    print(name, "chunk count:", len(splits))

recursive chunk count: 79
semantic chunk count: 72


In [8]:
EMBEDDING_NAME = "bge_m3"
chroma_root = PROJECT_DIR / "chroma_db"
vectorstores = {}

for name, splits in splits_by_method.items():
    collection_name = f"{EMBEDDING_NAME}_{name}"
    persist_directory = chroma_root / f"{EMBEDDING_NAME}_{name}"

    # Chunking veya splits değiştiyse existing index yeniden oluşturulmalıdır.
    if persist_directory.exists():
        vectorstores[name] = Chroma(
            collection_name=collection_name,
            persist_directory=str(persist_directory),
            embedding_function=embeddings
        )
    else:
        vectorstores[name] = Chroma.from_documents(
            documents=splits,
            embedding=embeddings,
            collection_name=collection_name,
            persist_directory=str(persist_directory)
        )

    count = vectorstores[name]._collection.count()
    expected_count = len(splits_by_method[name])
    if count != expected_count:
        raise ValueError(f"{name} count {count}, beklenen {expected_count}")
    print(name, "count:", count)

recursive count: 79
semantic count: 72


## Candidate retrieval and reranking

`BAAI/bge-reranker-v2-m3` is a multilingual cross-encoder that works well with BGE-M3 retrieval. It is larger than MiniLM, but still practical for 15 candidates per question on a 16 GB MacBook. The same model and parameters are used for both chunking methods.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import torch
from sentence_transformers import CrossEncoder

CANDIDATE_K = 15
FINAL_K = 5
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"

reranker_device = "mps" if torch.backends.mps.is_available() else "cpu"
reranker = CrossEncoder(
    RERANKER_MODEL,
    max_length=512,
    device=reranker_device
)

retrieval_questions = json.loads(
    (PROJECT_DIR / "eval_questions_retrieval.json").read_text(encoding="utf-8")
)

reranked_rows = []

for item in retrieval_questions:
    question_id = item["id"]
    question = item["question"]

    for method in ("recursive", "semantic"):
        candidate_hits = vectorstores[method].similarity_search_with_score(
            question,
            k=CANDIDATE_K
        )

        pairs = [(question, doc.page_content) for doc, _ in candidate_hits]
        reranker_scores = reranker.predict(
            pairs,
            batch_size=8,
            show_progress_bar=False
        )

        candidates = []
        for original_rank, ((doc, _), score) in enumerate(
            zip(candidate_hits, reranker_scores),
            start=1
        ):
            metadata = doc.metadata or {}
            candidates.append({
                "method": method,
                "question_id": question_id,
                "original_rank": original_rank,
                "chunk_id": metadata.get("chunk_id"),
                "reranker_score": float(score),
                "source": Path(metadata.get("source", "")).name,
                "page": metadata.get("page"),
                "chunk_text": doc.page_content,
            })

        candidates.sort(key=lambda row: row["reranker_score"], reverse=True)

        for reranked_rank, row in enumerate(candidates[:FINAL_K], start=1):
            row["reranked_rank"] = reranked_rank
            reranked_rows.append(row)

reranked_df = pd.DataFrame(reranked_rows, columns=[
    "method", "question_id", "original_rank", "reranked_rank",
    "chunk_id", "reranker_score", "source", "page", "chunk_text"
])

reranked_output_path = PROJECT_DIR / "results" / "reranked_top4.csv"
reranked_df.to_csv(reranked_output_path, index=False)

print(
    f"{len(reranked_df)} final chunk hazırlandı "
    f"({len(retrieval_questions)} soru, iki yöntem, Top-{FINAL_K})."
)
print(f"Sonuç: {reranked_output_path.name} | Model: {RERANKER_MODEL}")

example_question_id = retrieval_questions[0]["id"]
example_df = reranked_df[
    reranked_df["question_id"] == example_question_id
].sort_values(["method", "reranked_rank"])
display(example_df)

### Pipeline

BGE-M3 + Chroma → Top 15 candidates → Cross-Encoder Reranker → Top 4

## Manual gold-evidence check

This cell writes five questions, their gold evidence, and the reranker's Top-5 chunks to a readable text file. It does not compare texts or calculate any metric.

In [ ]:
manual_questions = retrieval_questions[:5]
manual_lines = []

for item in manual_questions:
    question_id = item["id"]
    manual_lines.append("=" * 50)
    manual_lines.append(question_id)
    manual_lines.append("QUESTION:")
    manual_lines.append(item["question"])
    manual_lines.append("")
    manual_lines.append("GOLD EVIDENCE:")

    for evidence_index, evidence in enumerate(item["gold_evidence"], start=1):
        manual_lines.append(
            f"{evidence_index}. [{evidence['source']}, page {evidence['page']}]"
        )
        manual_lines.append(evidence["text"])
        manual_lines.append("")

    for method in ("recursive", "semantic"):
        manual_lines.append(f"{method.upper()} TOP-5:")
        method_rows = reranked_df[
            (reranked_df["question_id"] == question_id) &
            (reranked_df["method"] == method)
        ].sort_values("reranked_rank").head(5)

        for _, row in method_rows.iterrows():
            manual_lines.append(
                f"{int(row['reranked_rank'])}. [{row['chunk_id']}] "
                f"[{row['source']}, page {row['page']}] "
                f"score={row['reranker_score']:.4f} "
                f"(original rank {int(row['original_rank'])})"
            )
            manual_lines.append(row["chunk_text"])
            manual_lines.append("")

    manual_lines.append("")

manual_check_path = PROJECT_DIR / "results" / "gold_evidence_chunk_check.txt"
manual_check_path.write_text("\n".join(manual_lines), encoding="utf-8")
print(f"Manuel kontrol dosyası hazır: {manual_check_path.name}")

In [10]:
import time
import pandas as pd
import ollama

# --------------------------------------------------
# Benchmark settings
# --------------------------------------------------

BENCHMARK_MODELS = [
    "qwen2.5:3b",
    "qwen2.5:7b",
]

BENCHMARK_QUESTION_COUNT = 3
BENCHMARK_METHOD = "recursive"

PROMPT_TEMPLATE = """
You are a question-answering assistant.

Answer the question using ONLY the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not add information that is not supported by the context.
3. Answer every part of the question.
4. If the question has multiple parts, evaluate each part separately.
5. If one part is supported and another is not, answer the supported
   part and explicitly state that the unsupported part is not specified.
6. Do not assume that the premise of the question is correct.
   If the context contradicts the premise, explicitly correct it.
7. When comparing multiple entities or systems, keep their properties
   separate.
8. If the requested information is completely absent, say that it is
   not specified in the provided context.
9. Do not estimate or speculate about missing numerical values.
10. Keep the answer focused on the question.

Context:
{context}

Question:
{question}

Answer:
""".strip()


# --------------------------------------------------
# Warm-up
# Model loading time is not included in benchmark.
# --------------------------------------------------

for model in BENCHMARK_MODELS:
    print(f"{model} warm-up...")

    ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": "Reply only with: ready"
            }
        ],
        options={
            "temperature": 0
        }
    )

print("Warm-up tamamlandı.\n")


# --------------------------------------------------
# Benchmark
# --------------------------------------------------

benchmark_rows = []

benchmark_questions = retrieval_questions[:BENCHMARK_QUESTION_COUNT]

for item in benchmark_questions:

    question_id = item["id"]
    question = item["question"]

    # Recursive reranked Top-5 chunks
    context_rows = (
        reranked_df[
            (reranked_df["question_id"] == question_id) &
            (reranked_df["method"] == BENCHMARK_METHOD)
        ]
        .sort_values("reranked_rank")
        .head(FINAL_K)
    )

    # Safety check
    if len(context_rows) != FINAL_K:
        raise ValueError(
            f"{question_id} için {FINAL_K} chunk bekleniyordu, "
            f"{len(context_rows)} bulundu."
        )

    # Keep chunks clearly separated in the prompt
    contexts = context_rows["chunk_text"].tolist()

    context_text = "\n\n".join(
        f"[Context {i}]\n{text}"
        for i, text in enumerate(contexts, start=1)
    )

    prompt = PROMPT_TEMPLATE.format(
        context=context_text,
        question=question
    )

    # Same prompt/context for both models
    for model in BENCHMARK_MODELS:

        print(f"{question_id} | {model}")

        start_time = time.perf_counter()

        response = ollama.chat(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ],
            options={
                "temperature": 0
            }
        )

        end_time = time.perf_counter()

        generation_time = end_time - start_time
        answer = response["message"]["content"].strip()

        benchmark_rows.append({
            "question_id": question_id,
            "model": model,
            "generation_time_seconds": generation_time,
            "answer": answer
        })


# --------------------------------------------------
# Results
# --------------------------------------------------

benchmark_df = pd.DataFrame(benchmark_rows)

display(benchmark_df)


# --------------------------------------------------
# Timing summary
# --------------------------------------------------

benchmark_summary = (
    benchmark_df
    .groupby("model")["generation_time_seconds"]
    .agg(
        total_time_seconds="sum",
        average_time_seconds="mean"
    )
    .reset_index()
)

print("\nTiming Summary:")
display(benchmark_summary)


# --------------------------------------------------
# Save results
# --------------------------------------------------

results_dir = PROJECT_DIR / "results"
results_dir.mkdir(parents=True, exist_ok=True)

benchmark_output_path = (
    results_dir / "generation_model_benchmark.csv"
)

benchmark_df.to_csv(
    benchmark_output_path,
    index=False
)

print(f"Benchmark kaydedildi: {benchmark_output_path}")

qwen2.5:3b warm-up...
qwen2.5:7b warm-up...
Warm-up tamamlandı.

Q01 | qwen2.5:3b
Q01 | qwen2.5:7b
Q02 | qwen2.5:3b
Q02 | qwen2.5:7b
Q03 | qwen2.5:3b
Q03 | qwen2.5:7b


,question_id,model,generation_time_seconds,answer
0,Q01,qwen2.5:3b,16.802139,Artificial intelligence (AI) improves smart gr...
1,Q01,qwen2.5:7b,19.598368,Artificial intelligence improves smart grid ma...
2,Q02,qwen2.5:3b,9.047123,"The main advantages of sodium-ion batteries, a..."
3,Q02,qwen2.5:7b,16.202273,"The main advantages of sodium-ion batteries, a..."
4,Q03,qwen2.5:3b,10.261423,Perovskite-Silicon Tandem Solar Cells improve ...
5,Q03,qwen2.5:7b,17.054764,Perovskite-Silicon tandem solar cells improve ...



Timing Summary:


,model,total_time_seconds,average_time_seconds
0,qwen2.5:3b,36.110685,12.036895
1,qwen2.5:7b,52.855405,17.618468


Benchmark kaydedildi: /Users/azatdundar/Desktop/DeepLearning/ragas_project/results/generation_model_benchmark.csv


## Final answer generation

This cell uses the existing reranked Top-5 chunks. It does not run retrieval, embedding, Chroma indexing, reranking, or RAGAS.

In [11]:
import json
import time
import pandas as pd
from langchain_ollama import ChatOllama

GENERATION_MODEL = "qwen2.5:7b"
TEMPERATURE = 0

generation_llm = ChatOllama(
    model=GENERATION_MODEL,
    temperature=TEMPERATURE
)

generation_prompt = """You are a question-answering assistant.

Answer the question using ONLY the provided context.

Rules:

1. Do not use outside knowledge.
2. Do not add information that is not supported by the context.
3. Answer every part of the question.
4. If the question has multiple parts, evaluate each part separately.
5. If one part is supported and another is not, answer the supported
   part and explicitly state that the unsupported part is not specified.
6. Do not assume that the premise of the question is correct.
   If the context contradicts the premise, explicitly correct it.
7. When comparing multiple entities or systems, keep their properties
   separate. Do not attribute information from one entity to another.
8. If the requested information is completely absent, say that it is
   not specified in the provided context.
9. Do not estimate or speculate about missing numerical values.
10. Keep the answer focused on the question.

Context:
{context}

Question:
{question}

Answer:
"""

generation_questions = json.loads(
    (PROJECT_DIR / "eval_questions_retrieval.json").read_text(encoding="utf-8")
)
ground_truth_by_question = {
    item["id"]: item["ground_truth"] for item in generation_questions
}

generated_rows = []
total_generation_start = time.perf_counter()

for item in generation_questions:
    question_id = item["id"]
    question = item["question"]

    for method in ("recursive", "semantic"):
        selected = reranked_df[
            (reranked_df["question_id"] == question_id) &
            (reranked_df["method"] == method)
        ].sort_values("reranked_rank").head(5)

        if len(selected) != 5:
            raise ValueError(
                f"{question_id} / {method} için 5 reranked chunk bulunamadı."
            )

        contexts = selected["chunk_text"].tolist()
        context_chunk_ids = selected["chunk_id"].tolist()
        context = "\n\n".join(
            f"[Context {index}]\n{text}"
            for index, text in enumerate(contexts, start=1)
        )

        prompt = generation_prompt.format(
            context=context,
            question=question
        )

        start_time = time.perf_counter()
        answer = generation_llm.invoke(prompt).content
        generation_time = time.perf_counter() - start_time

        generated_rows.append({
            "question_id": question_id,
            "method": method,
            "question": question,
            "contexts": json.dumps(contexts, ensure_ascii=False),
            "context_chunk_ids": json.dumps(context_chunk_ids, ensure_ascii=False),
            "answer": answer,
            "ground_truth": ground_truth_by_question[question_id],
            "generation_model": GENERATION_MODEL,
            "generation_time_seconds": generation_time,
        })

generated_df = pd.DataFrame(generated_rows, columns=[
    "question_id", "method", "question", "contexts",
    "context_chunk_ids", "answer", "ground_truth",
    "generation_model", "generation_time_seconds"
])

assert len(generated_df) == 46
expected_pairs = {
    (item["id"], method)
    for item in generation_questions
    for method in ("recursive", "semantic")
}
actual_pairs = set(zip(generated_df["question_id"], generated_df["method"]))
assert actual_pairs == expected_pairs
assert generated_df.groupby(["question_id", "method"]).size().eq(1).all()

output_path = PROJECT_DIR / "results" / "generated_answers_top5.csv"
generated_df.to_csv(output_path, index=False)

total_generation_time = time.perf_counter() - total_generation_start
average_generation_time = generated_df["generation_time_seconds"].mean()

print(f"{len(generated_df)} cevap oluşturuldu: {output_path.name}")
print(f"Toplam süre: {total_generation_time:.1f} saniye | Ortalama: {average_generation_time:.1f} saniye")
display(generated_df.head())

46 cevap oluşturuldu: generated_answers_top5.csv
Toplam süre: 707.1 saniye | Ortalama: 15.4 saniye


,question_id,method,question,contexts,context_chunk_ids,answer,ground_truth,generation_model,generation_time_seconds
0,Q01,recursive,How does artificial intelligence improve smart...,"[""11 Conclusion\nSmart grids represent a funda...","[""recursive-0051"", ""recursive-0038"", ""recursiv...",Artificial intelligence improves smart grid ma...,Artificial intelligence improves smart grid ma...,qwen2.5:7b,23.210500
1,Q01,semantic,How does artificial intelligence improve smart...,"[""11 Conclusion\nSmart grids represent a funda...","[""semantic-0048"", ""semantic-0036"", ""semantic-0...",Artificial intelligence improves smart grid ma...,Artificial intelligence improves smart grid ma...,qwen2.5:7b,19.283323
2,Q02,recursive,What are the main advantages of sodium-ion bat...,"[""applications.\nTheselimitationshaveencourage...","[""recursive-0006"", ""recursive-0007"", ""recursiv...","The main advantages of sodium-ion batteries, a...",Sodium-ion batteries offer lower raw material ...,qwen2.5:7b,10.532877
3,Q02,semantic,What are the main advantages of sodium-ion bat...,"[""•Enhanced low-temperature performance. Becau...","[""semantic-0006"", ""semantic-0004"", ""semantic-0...","The main advantages of sodium-ion batteries, a...",Sodium-ion batteries offer lower raw material ...,qwen2.5:7b,10.032707
4,Q03,recursive,How do Perovskite-Silicon tandem solar cells i...,"[""gies. Perovskite-based solar cells have attr...","[""recursive-0018"", ""recursive-0055"", ""recursiv...",Perovskite-Silicon tandem solar cells improve ...,They combine a perovskite layer with a silicon...,qwen2.5:7b,11.757065


## RAGAS evaluation

This cell evaluates the existing generated answers only. It does not run retrieval, reranking, embedding, or answer generation.

In [ ]:
import json
import pandas as pd
from langchain_ollama import ChatOllama, OllamaEmbeddings

EVALUATOR_LLM = "qwen2.5:3b"
EVALUATOR_EMBEDDING = "bge-m3"

generated_path = PROJECT_DIR / "results" / "generated_answers_top5.csv"
generated_input_df = pd.read_csv(generated_path)
generated_input_df["contexts"] = generated_input_df["contexts"].apply(json.loads)

if len(generated_input_df) != 46:
    raise ValueError(f"Beklenen 46 satır, bulunan {len(generated_input_df)}")
counts = generated_input_df["method"].value_counts().to_dict()
if counts != {"recursive": 23, "semantic": 23}:
    raise ValueError(f"Beklenmeyen method sayıları: {counts}")
required_columns = ["question", "contexts", "answer", "ground_truth"]
if generated_input_df[required_columns].isna().any().any():
    raise ValueError("Gerekli alanlarda eksik veri var.")

records = generated_input_df.rename(columns={
    "question": "user_input",
    "contexts": "retrieved_contexts",
    "answer": "response",
    "ground_truth": "reference"
})[["user_input", "retrieved_contexts", "response", "reference"]].to_dict(orient="records")

evaluator_llm = ChatOllama(model=EVALUATOR_LLM, temperature=0)
evaluator_embeddings = OllamaEmbeddings(model=EVALUATOR_EMBEDDING)

from ragas import EvaluationDataset, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.run_config import RunConfig
from ragas.metrics import Faithfulness, LLMContextPrecisionWithReference, LLMContextRecall, ResponseRelevancy

evaluation_dataset = EvaluationDataset.from_list(records)
wrapped_llm = LangchainLLMWrapper(evaluator_llm)
wrapped_embeddings = LangchainEmbeddingsWrapper(evaluator_embeddings)
run_config = RunConfig(timeout=300, max_retries=3, max_wait=60, max_workers=1)
evaluation_metrics = [
    Faithfulness(llm=wrapped_llm),
    ResponseRelevancy(llm=wrapped_llm, embeddings=wrapped_embeddings),
    LLMContextPrecisionWithReference(llm=wrapped_llm),
    LLMContextRecall(llm=wrapped_llm),
]

evaluation_result = evaluate(
    evaluation_dataset,
    metrics=evaluation_metrics,
    run_config=run_config,
    raise_exceptions=False
)

scores_df = evaluation_result.to_pandas()
if "response_relevancy" in scores_df.columns:
    scores_df["answer_relevancy"] = scores_df["response_relevancy"]
if "llm_context_precision_with_reference" in scores_df.columns:
    scores_df["context_precision"] = scores_df["llm_context_precision_with_reference"]
elif "context_precision_with_reference" in scores_df.columns:
    scores_df["context_precision"] = scores_df["context_precision_with_reference"]

row_metadata = generated_input_df[["question_id", "method", "question"]].reset_index(drop=True)
scores_df = pd.concat([row_metadata, scores_df.reset_index(drop=True)], axis=1)
metric_columns = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
missing_metrics = [name for name in metric_columns if name not in scores_df.columns]
if missing_metrics:
    raise ValueError(f"Eksik RAGAS skorları: {missing_metrics}")

ragas_scores_df = scores_df[["question_id", "method", "question"] + metric_columns]
if len(ragas_scores_df) != 46:
    raise ValueError(f"RAGAS sonucu 46 satır olmalı, bulunan: {len(ragas_scores_df)}")

scores_output_path = PROJECT_DIR / "results" / "ragas_scores_top5.csv"
ragas_scores_df.to_csv(scores_output_path, index=False)

ragas_summary_df = ragas_scores_df.groupby("method")[metric_columns].mean().reset_index()
summary_output_path = PROJECT_DIR / "results" / "ragas_summary_top5.csv"
ragas_summary_df.to_csv(summary_output_path, index=False)

ragas_std_df = ragas_scores_df.groupby("method")[metric_columns].std().reset_index()
print("NaN skorları:")
print(ragas_scores_df[metric_columns].isna().sum())
display(ragas_summary_df)
display(ragas_std_df)


In [18]:
import json
import pandas as pd
from langchain_ollama import ChatOllama, OllamaEmbeddings
from ragas import EvaluationDataset, evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, LLMContextPrecisionWithReference, LLMContextRecall, ResponseRelevancy
from ragas.run_config import RunConfig

pilot_input_df = pd.read_csv(PROJECT_DIR / "results" / "generated_answers_top5.csv")
pilot_input_df["contexts"] = pilot_input_df["contexts"].apply(json.loads)
pilot_input_df = pd.concat([
    pilot_input_df[pilot_input_df["method"] == "recursive"].head(5),
    pilot_input_df[pilot_input_df["method"] == "semantic"].head(5)
]).reset_index(drop=True)

if len(pilot_input_df) != 10:
    raise ValueError(f"Pilot için beklenen 10 satır, bulunan: {len(pilot_input_df)}")

pilot_records = pilot_input_df.rename(columns={
    "question": "user_input",
    "contexts": "retrieved_contexts",
    "answer": "response",
    "ground_truth": "reference"
})[["user_input", "retrieved_contexts", "response", "reference"]].to_dict(orient="records")

pilot_llm = LangchainLLMWrapper(ChatOllama(model="qwen2.5:7b", temperature=0))
pilot_embeddings = LangchainEmbeddingsWrapper(OllamaEmbeddings(model="bge-m3"))
pilot_metrics = [
    Faithfulness(llm=pilot_llm),
    ResponseRelevancy(llm=pilot_llm, embeddings=pilot_embeddings),
    LLMContextPrecisionWithReference(llm=pilot_llm),
    LLMContextRecall(llm=pilot_llm),
]

pilot_result = evaluate(
    EvaluationDataset.from_list(pilot_records),
    metrics=pilot_metrics,
    run_config=RunConfig(timeout=300, max_retries=3, max_wait=60, max_workers=1),
    raise_exceptions=False
)

pilot_scores_df = pilot_result.to_pandas()
if "response_relevancy" in pilot_scores_df.columns:
    pilot_scores_df["answer_relevancy"] = pilot_scores_df["response_relevancy"]
if "llm_context_precision_with_reference" in pilot_scores_df.columns:
    pilot_scores_df["context_precision"] = pilot_scores_df["llm_context_precision_with_reference"]
elif "context_precision_with_reference" in pilot_scores_df.columns:
    pilot_scores_df["context_precision"] = pilot_scores_df["context_precision_with_reference"]

pilot_metadata = pilot_input_df[["question_id", "method", "question"]].reset_index(drop=True)
pilot_scores_df = pd.concat([pilot_metadata, pilot_scores_df.reset_index(drop=True)], axis=1)
pilot_metric_columns = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
pilot_scores_df = pilot_scores_df[["question_id", "method", "question"] + pilot_metric_columns]

pilot_scores_path = PROJECT_DIR / "results" / "ragas_scores_pilot10.csv"
pilot_summary_path = PROJECT_DIR / "results" / "ragas_summary_pilot10.csv"
pilot_scores_df.to_csv(pilot_scores_path, index=False)
pilot_summary_df = pilot_scores_df.groupby("method")[pilot_metric_columns].mean().reset_index()
pilot_summary_df.to_csv(pilot_summary_path, index=False)
display(pilot_summary_df)
print("RAGAS pilot tamamlandı: 10 örnek değerlendirildi.")

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

,method,faithfulness,answer_relevancy,context_precision,context_recall
0,recursive,0.910000,0.958400,0.953333,1.000000
1,semantic,0.971429,0.959766,0.917500,0.966667


RAGAS pilot tamamlandı: 10 örnek değerlendirildi.
